In [1]:
# ---------------------------------------------------------------------------
# INSTALL com uv - wheel CUDA pre-compilado (NADA e compilado)
# ---------------------------------------------------------------------------
# O notebook usa a API in-process do llama-cpp-python: uvicorn/servidor HTTP
# NAO sao necessarios aqui (so seriam para o modo servidor OpenAI-compativel).
#
# 1) Instala o uv (instalador rapido em Rust) via pip - pacote pequeno.
# 2) numpy <2.3: evita o conflito com o numba 0.61.2 pre-instalado no Colab.
# 3) llama-cpp-python CUDA (cu125) vem do release oficial do autor, como
#    wheel pronto: sem CMAKE_ARGS, sem FORCE_CMAKE, sem build de ~30-40 min.
#    T4 (compute capability 7.5) e suportada por wheels cu12x.
#
# Se o Colab migrar para CUDA 13, troque o URL abaixo para o release -cu130
# equivalente (mesmo padrao: v0.3.35-cu130/llama_cpp_python-0.3.35-...whl).

!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu125
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 707.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [2]:
# Create the target directory
!mkdir -p /content/models

# Download the model weights directly using huggingface_hub
!pip install -q huggingface_hub
!python3 -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='bartowski/Qwen_Qwen3.5-0.8B-GGUF', filename='Qwen_Qwen3.5-0.8B-Q4_K_L.gguf', local_dir='/content/models')"


Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:   1% 4.47M/641M [00:00<02:16, 4.66MB/s]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  38% 241M/641M [00:01<00:01, 348MB/s, 18.2MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  45% 289M/641M [00:01<00:01, 296MB/s, 24.1MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  60% 385M/641M [00:02<00:01, 229MB/s, 33.1MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: reconstructing file:  59% 381M/641M [00:02<00:01, 199MB/s, 26.4MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  81% 519M/641M [00:03<00:00, 196MB/s, 42.4MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  88% 567M/641M [00:03<00:00, 209MB/s, 45.9MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes:  94% 605M/641M [00:03<00:00, 202MB/s, 49.2MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: downloading bytes: 100% 634M/634M [00:03<00:00, 167MB/s, 52.2MB/s  ]
Qwen_Qwen3.5-0.8B-Q4_K_L.gguf: reconstructing file: 100% 641M/641M [00:03<00:00, 169MB/s, 54.1MB/s  ]
mv: c

In [3]:
# ---------------------------------------------------------------------------
# MODELO QWEN GGUF - contexto 32k com KV quantizado
# ---------------------------------------------------------------------------
import os
import shutil

from llama_cpp import Llama

# Caminho do GGUF (padrao: /content). Se estiver no Google Drive (FUSE),
# copie para /content antes do load: mmap sobre o Drive deixa o acesso aos
# pesos lento (page faults por arquivo remoto).
MODEL_PATH = "/content/models/Qwen_Qwen3.5-0.8B-Q4_K_L.gguf"

_LOCAL_PATH = "/content/models_local/Qwen_Qwen3.5-0.8B-Q4_K_L.gguf"
if str(MODEL_PATH).startswith("/content/drive/"):
    os.makedirs(os.path.dirname(_LOCAL_PATH), exist_ok=True)
    if not os.path.exists(_LOCAL_PATH):
        print("Copiando GGUF do Drive para /content (1x)...")
        shutil.copyfile(MODEL_PATH, _LOCAL_PATH)
        print("Copia concluida.")
    MODEL_PATH = _LOCAL_PATH

# n_ctx    : 32768 -> documentos de ate ~30k tokens processados inteiros.
# type_k/v : "q8_0" -> KV cache quantizado: 32k tokens cabem na T4 16GB.
# n_batch  : tokens por passo de GPU (512 satura bem a T4).
# ATENCAO  : nunca chame llm.reset() entre chamadas. O "input cache" depende
#            do prefixo continuar no KV cache da sessao.
try:
    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=32768,
        n_batch=512,
        n_ubatch=512,
        n_gpu_layers=-1,
        type_k="q8_0",
        type_v="q8_0",
        verbose=False,
        seed=0,
    )
except TypeError:
    print("AVISO: KV quantizado (q8_0) nao suportado nesta versao; usando f16.")
    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=32768,
        n_batch=512,
        n_gpu_layers=-1,
        verbose=False,
        seed=0,
    )

print("Llama model loaded successfully.")
print("n_ctx:", llm.n_ctx())

# Opcional - equivalente local ao prompt cache de API entre sessoes:
# from llama_cpp.llama_cache import LlamaDiskCache
# llm.set_cache(LlamaDiskCache(cache_dir="/content/llama_state_cache"))


AVISO: KV quantizado (q8_0) nao suportado nesta versao; usando f16.
Llama model loaded successfully.
n_ctx: 32768


In [4]:
# ===========================================================================
# CLASSIFICACAO: INPUT CACHE + STRUCTURED OUTPUT (JSON Schema / gramatica)
# ===========================================================================
# INPUT CACHE (equivalente local ao "prompt caching" das APIs de LLM):
#   PROMPT_SISTEMA (persona + rubricas) e o BLOCO CACHEAVEL: bytes identicos
#   em todas as chamadas. O llama.cpp mantem esse prefixo no KV cache e nas
#   chamadas seguintes NAO o reavalia (input cache hit). A 2a/3a tarefa do
#   mesmo processo tambem reusam o prefixo que contem o documento.
#
# STRUCTURED OUTPUT:
#   NENHUMA instrucao textual de formato existe no prompt. O formato e
#   garantido pelos JSON Schemas abaixo, convertidos em gramatica GBNF e
#   aplicados na amostragem (constrained sampling).
# ===========================================================================
import json
import time

from llama_cpp.llama_grammar import LlamaGrammar

# ---------------------------------------------------------------------------
# Schemas - definem o formato (nao ha "responda apenas com o digito")
# ---------------------------------------------------------------------------
SCHEMA_BINARIO = {
    "type": "object",
    "properties": {"resultado": {"type": "integer", "enum": [0, 1]}},
    "required": ["resultado"],
}

SCHEMA_CLASSIFICACAO = {
    "type": "object",
    "properties": {"resultado": {"type": "integer", "enum": [1, 2, 3]}},
    "required": ["resultado"],
}

# ---------------------------------------------------------------------------
# BLOCO CACHEAVEL (mensagem system) - mesmo texto em TODA a execucao
# ---------------------------------------------------------------------------
PROMPT_SISTEMA = """Você é um especialista em provas digitais no processo judicial brasileiro e um analista criterioso de textos processuais.

Você recebe o texto integral de um processo judicial e, em seguida, perguntas objetivas numeradas como TAREFA 1, TAREFA 2 e TAREFA 3.

REGRAS GERAIS OBRIGATÓRIAS:
1. Não faça inferências.
2. Considere exclusivamente as informações expressamente presentes no texto fornecido.
3. Responda cada tarefa de forma independente, aplicando somente as regras da tarefa correspondente.
4. A saída de cada tarefa é um objeto JSON cujo formato é definido pelo schema correspondente.

TAREFA 1 - EXISTÊNCIA DE PROVA DIGITAL:
Determinar se existe PROVA DIGITAL relacionada aos fatos discutidos no processo.
Considere apenas evidências digitais utilizadas para demonstrar, confirmar, refutar ou contextualizar os fatos controvertidos.
Considere como prova digital:
- mensagens de WhatsApp, Telegram, SMS ou similares;
- e-mails;
- capturas de tela (prints);
- fotografias digitais;
- vídeos;
- áudios;
- gravações;
- publicações em redes sociais;
- registros de sistemas;
- logs;
- metadados;
- dados extraídos de celulares, computadores ou outros dispositivos;
- arquivos eletrônicos apresentados como evidência dos fatos;
- conteúdo armazenado em serviços digitais.
NÃO considere como prova digital:
- processo eletrônico;
- petições eletrônicas;
- movimentações processuais;
- documentos assinados digitalmente;
- certificados digitais;
- assinaturas eletrônicas;
- intimações eletrônicas;
- documentos meramente digitalizados;
- referências ao sistema do tribunal;
- e-SAJ, PJe, Projudi ou sistemas equivalentes;
- atos processuais eletrônicos em geral.
IMPORTANTE:
A mera existência de documentos eletrônicos nos autos NÃO significa existência de prova digital.
Classifique como 1 somente quando houver evidência digital relacionada aos fatos discutidos no processo.
Escala da TAREFA 1:
0 = Não há evidência de prova digital relacionada aos fatos.
1 = Há evidência de prova digital relacionada aos fatos.

TAREFA 2 - IMPUGNAÇÃO ESPECÍFICA DA PROVA DIGITAL:
Determinar se alguma das partes apresentou impugnação específica contra uma prova digital.
Considere apenas manifestações expressas que questionem a própria confiabilidade da prova digital.
Exemplos de impugnação específica:
- questionamento da autenticidade;
- questionamento da integridade;
- questionamento da origem;
- questionamento da autoria;
- alegação de adulteração;
- alegação de manipulação;
- alegação de montagem;
- alegação de edição;
- alegação de ausência ou ruptura da cadeia de custódia;
- questionamento dos métodos de coleta ou extração;
- questionamento de metadados, logs ou hashes;
- questionamento da confiabilidade técnica da evidência digital.
Considere petições, manifestações, recursos, quesitos, pareceres técnicos ou outros documentos.
NÃO considere como impugnação específica:
- mera discordância sobre os fatos;
- negativa dos fatos alegados;
- alegação genérica de insuficiência probatória;
- alegação de falta de convencimento do juiz;
- alegação de que a prova não comprova determinada narrativa;
- discussão jurídica sem questionamento da confiabilidade da prova digital;
- pedido genérico de produção de provas.
IMPORTANTE:
A crítica ao conteúdo da prova não é necessariamente impugnação da prova digital.
Classifique como 1 apenas quando houver questionamento expresso da confiabilidade, autenticidade, integridade, origem ou obtenção da evidência digital.
Escala da TAREFA 2:
0 = Não há impugnação específica de prova digital.
1 = Há impugnação específica de prova digital.

TAREFA 3 - ADERÊNCIA AOS PARÂMETROS DE CONFIABILIDADE:
Avaliar a aderência da prova digital aos parâmetros de confiabilidade.
Analise exclusivamente as informações presentes no texto e avalie, quando aplicável:
- cadeia de custódia;
- origem e identificação da evidência;
- coleta ou extração;
- datas e responsáveis;
- preservação e armazenamento;
- transferências e acessos;
- cópias e análises;
- hashes e integridade;
- imagens forenses;
- metadados e logs;
- autenticidade;
- método e ferramentas utilizadas;
- documentação;
- auditabilidade;
- contraditório;
- inconsistências entre documentos.
REGRAS OBRIGATÓRIAS DA TAREFA 3:
1. Não faça inferências.
2. Considere apenas informações expressamente presentes no texto.
3. A ausência de documentação NÃO significa automaticamente descumprimento.
4. Porém, a ausência de indícios negativos também NÃO significa aderência.
5. Classifique como 1 apenas quando existirem elementos positivos expressamente descritos nos autos que demonstrem aderência aos parâmetros.
6. Não utilize presunções de regularidade.
7. Quando as informações necessárias para avaliação forem insuficientes, limitadas ou inconclusivas, classifique como 3.
8. Diferencie "não consta dos autos" de "foi demonstrado que não foi realizado".
9. Para classificar como 2, deve existir evidência objetiva de fragilidade, descumprimento, inconsistência relevante ou questionamento fundamentado.
Escala da TAREFA 3:
1 = Potencialmente Segue. Existem evidências positivas e suficientes de aderência aos parâmetros relevantes para a prova digital analisada.
2 = Potencialmente Não Segue. Existe evidência objetiva de descumprimento, fragilidade relevante, inconsistência ou comprometimento da confiabilidade da prova digital.
3 = Indecisivo. As informações presentes são insuficientes, limitadas ou inconclusivas para avaliar aderência ou descumprimento."""

# Perguntas curtas (o documento e o prefixo comum entre elas no user message)
PERGUNTA_1 = "TAREFA 1: existe prova digital relacionada aos fatos discutidos no processo?"
PERGUNTA_2 = "TAREFA 2: alguma das partes apresentou impugnacao especifica contra uma prova digital?"
PERGUNTA_3 = "TAREFA 3: qual a classificacao de aderencia da prova digital aos parametros de confiabilidade?"

MAX_RESPONSE_TOKENS = 192
RESERVA_CONTEXTO = 256
CHUNK_MAX_TOKENS = 12000
CHUNK_OVERLAP_TOKENS = 400

_OVERHEAD_TOKENS = None


def _tokenizar(texto):
    return llm.tokenize(texto.encode("utf-8"), add_bos=False)


def _texto(tokens):
    return llm.detokenize(tokens).decode("utf-8", errors="replace")


def _overhead_tokens():
    """Tokens fixos por chamada: PROMPT_SISTEMA + wrapper/template/pergunta."""
    global _OVERHEAD_TOKENS
    if _OVERHEAD_TOKENS is None:
        _OVERHEAD_TOKENS = (
            len(_tokenizar(PROMPT_SISTEMA))
            + len(_tokenizar("TEXTO INTEGRAL DO PROCESSO:"))
            + 128
        )
    return _OVERHEAD_TOKENS


def _orcamento_do_documento():
    """Tokens disponiveis para o corpo do documento em cada chamada."""
    return llm.n_ctx() - MAX_RESPONSE_TOKENS - RESERVA_CONTEXTO - _overhead_tokens()


def _extrair_json(content):
    try:
        return json.loads(content.strip())
    except (ValueError, TypeError):
        pass
    ini = content.find("{")
    fim = content.rfind("}")
    if ini != -1 and fim > ini:
        return json.loads(content[ini:fim + 1])
    raise ValueError(f"Resposta fora do schema (nao-JSON): {content!r}")


def _perguntar(pergunta, schema, documento):
    """Uma chamada estruturada.

    system = PROMPT_SISTEMA (fixo -> cacheavel em toda a execucao).
    user   = documento + pergunta curta (documento vira prefixo comum entre
    as 3 tarefas de um mesmo processo).
    Formato garantido pelo schema (gramatica GBNF), sem instrucao textual.
    """
    user_prompt = f"""TEXTO INTEGRAL DO PROCESSO:

{documento}

{pergunta}"""
    messages = [
        {"role": "system", "content": PROMPT_SISTEMA},
        {"role": "user", "content": user_prompt},
    ]
    try:
        response = llm.create_chat_completion(
            messages=messages,
            response_format={"type": "json_object", "schema": schema},
            max_tokens=MAX_RESPONSE_TOKENS,
            temperature=0.0,
            seed=0,
        )
    except TypeError:
        grammar = LlamaGrammar.from_json_schema(json.dumps(schema))
        response = llm.create_chat_completion(
            messages=messages,
            grammar=grammar,
            max_tokens=MAX_RESPONSE_TOKENS,
            temperature=0.0,
            seed=0,
        )
    conteudo = response["choices"][0]["message"]["content"]
    return int(_extrair_json(conteudo)["resultado"])


def _classificar_um_documento(documento):
    """3 tarefas sobre o MESMO documento (as tarefas 2 e 3 reusam o KV)."""
    resultado = {}
    tarefas = [
        ("MIDIAS_DIGITAIS", PERGUNTA_1, SCHEMA_BINARIO),
        ("IMPUGNACAO_DA_PROVA_DIGITAL", PERGUNTA_2, SCHEMA_BINARIO),
        ("CLASSIFICACAO", PERGUNTA_3, SCHEMA_CLASSIFICACAO),
    ]
    for chave, pergunta, schema in tarefas:
        try:
            resultado[chave] = _perguntar(pergunta, schema, documento)
        except Exception as e:
            print(f"    ! {chave}: {e}")
            resultado[chave] = None
    return resultado


def _montar_chunks(tokens, chunk_max=CHUNK_MAX_TOKENS, overlap=CHUNK_OVERLAP_TOKENS):
    total = len(tokens)
    if total <= chunk_max:
        return [tokens]
    partes, i = [], 0
    while i < total:
        j = min(i + chunk_max, total)
        partes.append(tokens[i:j])
        if j == total:
            break
        i = max(j - overlap, i + 1)
    return partes


def _agregar(parciais):
    """OR para binarios; pior caso (2 > 1 > 3) para a classificacao."""
    def _valores(chave):
        return [p[chave] for p in parciais if p.get(chave) is not None]

    medias = _valores("MIDIAS_DIGITAIS")
    impugs = _valores("IMPUGNACAO_DA_PROVA_DIGITAL")
    clss = _valores("CLASSIFICACAO")
    return {
        "MIDIAS_DIGITAIS": 1 if any(v == 1 for v in medias) else (0 if medias else None),
        "IMPUGNACAO_DA_PROVA_DIGITAL": 1 if any(v == 1 for v in impugs) else (0 if impugs else None),
        "CLASSIFICACAO": 2 if 2 in clss else (1 if 1 in clss else (3 if clss else None)),
    }


def classify_process_text(documento):
    """Classifica um processo. Retorna (media, impugnacao, classificacao, info).

    info: tokens, modo (direto|chunks), n_chunks, tempo_llm_s, motivo.
    """
    t0 = time.perf_counter()
    info = {"modo": "direto", "chunks": 1, "tokens": 0, "tempo_llm_s": 0.0, "motivo": None}
    try:
        tokens_doc = _tokenizar(documento)
    except Exception as e:
        info["motivo"] = f"tokenize falhou: {e}"
        return None, None, None, info

    info["tokens"] = len(tokens_doc)
    orcamento = _orcamento_do_documento()

    if info["tokens"] <= orcamento:
        parcial = _classificar_um_documento(documento)
    else:
        chunk_max = min(CHUNK_MAX_TOKENS, orcamento)
        partes = _montar_chunks(tokens_doc, chunk_max)
        info["modo"] = "chunks"
        info["chunks"] = len(partes)
        print(
            f"    (doc grande: {info['tokens']} tok -> "
            f"{len(partes)} chunks de ~{chunk_max} tok)"
        )
        resultados = []
        for k, parte in enumerate(partes, start=1):
            print(f"    chunk {k}/{len(partes)} ...")
            resultados.append(_classificar_um_documento(_texto(parte)))
        parcial = _agregar(resultados)

    info["tempo_llm_s"] = time.perf_counter() - t0
    if (parcial["MIDIAS_DIGITAIS"] is None
            and parcial["IMPUGNACAO_DA_PROVA_DIGITAL"] is None
            and parcial["CLASSIFICACAO"] is None):
        info["motivo"] = "todas as tarefas falharam"
    return (
        parcial["MIDIAS_DIGITAIS"],
        parcial["IMPUGNACAO_DA_PROVA_DIGITAL"],
        parcial["CLASSIFICACAO"],
        info,
    )


In [ ]:
import os
import time

import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    def tqdm(seq, **kw):
        return seq

# ---------------------------------------------------------------------------
# CONFIG / PASTAS / SAIDA
# ---------------------------------------------------------------------------
root_folder = "/content/drive/MyDrive/ocr_export"
output_csv_path = "/content/drive/MyDrive/process_classification_results.csv"
batch_size = 10

LINHA = "=" * 64
TRACO = "-" * 64

METRICAS = {
    "inicio": time.perf_counter(),
    "novos": 0,
    "feitos": 0,
    "ok": 0,
    "parciais": 0,
    "nulos": 0,
    "tokens_prompt": 0,
    "leitura_s": 0.0,
    "llm_s": 0.0,
    "chunks": 0,
}


def _ler_texto(pasta):
    partes = []
    for arquivo in sorted(os.listdir(pasta)):
        if not arquivo.lower().endswith(".txt"):
            continue
        caminho = os.path.join(pasta, arquivo)
        try:
            with open(caminho, "r", encoding="utf-8") as f:
                partes.append(f.read())
        except Exception as e:
            print(f"     ! erro ao ler {arquivo}: {e}")
    return "\n".join(partes)


def _formatar(media, imp, cls):
    return " | ".join("-" if v is None else str(v) for v in (media, imp, cls))


def _estimativas():
    decorrido = time.perf_counter() - METRICAS["inicio"]
    feitos = METRICAS["feitos"]
    restantes = max(METRICAS["novos"] - feitos, 0)
    taxa = feitos / decorrido if decorrido > 0 else 0.0
    eta_min = (restantes / taxa) / 60.0 if taxa > 0 else float("nan")
    tok_s = (
        METRICAS["tokens_prompt"] / METRICAS["llm_s"]
        if METRICAS["llm_s"] > 0 else float("nan")
    )
    return {"feitos": feitos, "restantes": restantes, "doc_min": taxa * 60.0,
            "eta_min": eta_min, "tok_s": tok_s}


def _resumo_batch():
    e = _estimativas()
    print(TRACO)
    print(
        f"  OK {METRICAS['ok']} | PARCIAL {METRICAS['parciais']} | "
        f"NULO {METRICAS['nulos']} | chunks {METRICAS['chunks']}"
    )
    print(
        f"  VELOCIDADE: {e['doc_min']:.2f} doc/min | prefill ~{e['tok_s']:.0f} tok/s | "
        f"ETA ~{e['eta_min']:.0f} min"
    )
    print(
        f"  ACUMULADO: leitura {METRICAS['leitura_s']:.0f}s | "
        f"llm {METRICAS['llm_s']:.0f}s"
    )
    print(TRACO)


if not os.path.exists(root_folder):
    print(f"ERROR: pasta nao encontrada:\n{root_folder}")
else:
    # ---------------------------------------------------------------
    # resume: resultados ja gravados no CSV
    # ---------------------------------------------------------------
    existing_df = pd.DataFrame()
    if os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path)
        print(f"Carregados {len(existing_df)} resultados existentes (resume).")

    if not existing_df.empty and "Process ID" in existing_df.columns:
        processados = set(existing_df["Process ID"].astype(str).tolist())
    else:
        processados = set()

    todas_pastas = [
        p for p in os.listdir(root_folder)
        if os.path.isdir(os.path.join(root_folder, p))
    ]
    novas = [p for p in todas_pastas if p not in processados]
    METRICAS["novos"] = len(novas)

    print(LINHA)
    print(
        f"  PROCESSOS: {len(todas_pastas)} total | {len(novas)} novos | "
        f"{len(todas_pastas) - len(novas)} ja feitos"
    )
    print(LINHA)

    if not novas:
        print("  Nada novo para processar. Encerrando.")

    total_batches = (len(novas) + batch_size - 1) // batch_size if novas else 0

    for i in range(0, len(novas), batch_size):
        batch_folders = novas[i:i + batch_size]
        resultados_batch = []
        num = i // batch_size + 1
        t_batch = time.perf_counter()

        print("\n" + LINHA)
        print(f"  BATCH {num}/{total_batches}  ({len(batch_folders)} processos)")
        print(LINHA)

        for folder in tqdm(batch_folders, desc=f"batch {num}", leave=False):
            t_doc0 = time.perf_counter()
            caminho = os.path.join(root_folder, folder)

            texto = _ler_texto(caminho)
            METRICAS["leitura_s"] += time.perf_counter() - t_doc0

            media = imp = cls = None
            n_tokens = 0
            n_chunks = 0
            modo = "vazio"
            motivo = None
            if texto.strip():
                try:
                    media, imp, cls, info = classify_process_text(texto)
                    METRICAS["llm_s"] += info.get("tempo_llm_s", 0.0)
                    n_tokens = info.get("tokens", 0)
                    n_chunks = info.get("chunks", 1)
                    METRICAS["tokens_prompt"] += n_tokens
                    METRICAS["chunks"] += n_chunks
                    modo = info.get("modo", "?")
                    motivo = info.get("motivo")
                except Exception as e:
                    modo = "erro"
                    motivo = str(e)
                    print(f"     ERRO inesperado: {e}")

            METRICAS["feitos"] += 1
            if media is None and imp is None and cls is None:
                METRICAS["nulos"] += 1
                simbolo = "X"
            elif media is None or imp is None or cls is None:
                METRICAS["parciais"] += 1
                simbolo = "!"
            else:
                METRICAS["ok"] += 1
                simbolo = "ok"

            resultados_batch.append({
                "Process ID": folder,
                "MIDIAS_DIGITAIS": media,
                "IMPUGNACAO_DA_PROVA_DIGITAL": imp,
                "CLASSIFICACAO": cls,
            })

            dt_doc = time.perf_counter() - t_doc0
            modo_txt = {
                "direto": "1x",
                "chunks": f"chunk x{n_chunks}",
                "vazio": "sem txt",
                "erro": "erro",
            }.get(modo, str(modo))
            if motivo:
                modo_txt += f" ({motivo})"
            print(
                f"  [{simbolo}] {folder[:38]:<38} "
                f"{_formatar(media, imp, cls):<11} "
                f"{dt_doc:6.1f}s  {n_tokens / 1000:6.1f}k tok  {modo_txt}"
            )

        if resultados_batch:
            df_batch = pd.DataFrame(resultados_batch)
            existe = os.path.exists(output_csv_path)
            df_batch.to_csv(
                output_csv_path,
                index=False,
                mode="a" if existe else "w",
                header=not existe,
            )
            print(f"\n  Batch salvo -> {output_csv_path}")

        print(f"  Tempo do batch {num}: {time.perf_counter() - t_batch:.1f}s")
        _resumo_batch()

    # ---------------------------------------------------------------
    # relatorio final
    # ---------------------------------------------------------------
    if os.path.exists(output_csv_path):
        final = pd.read_csv(output_csv_path)
        e = _estimativas()
        print("\n" + LINHA)
        print("  PROCESSAMENTO COMPLETO")
        print(
            f"  CSV: {output_csv_path} | linhas {len(final)} | "
            f"processos unicos {final['Process ID'].nunique()}"
        )
        print(LINHA)
        print("  Contagem CLASSIFICACAO (CSV acumulado):")
        print(final["CLASSIFICACAO"].value_counts(dropna=False).to_string())
        print(LINHA)
        tabela = pd.DataFrame([{
            "novos": e["feitos"],
            "ok": METRICAS["ok"],
            "parcial": METRICAS["parciais"],
            "nulo": METRICAS["nulos"],
            "tokens_prefill": METRICAS["tokens_prompt"],
            "tempo_llm_s": round(METRICAS["llm_s"], 1),
            "tempo_leitura_s": round(METRICAS["leitura_s"], 1),
            "chunks": METRICAS["chunks"],
            "doc/min": round(e["doc_min"], 2),
            "prefill tok/s": (None if e["tok_s"] != e["tok_s"] else round(e["tok_s"], 0)),
        }])
        print("  ESTATISTICAS DA EXECUCAO:")
        print(tabela.to_string(index=False))
        print(LINHA)
    else:
        print("  Nenhum CSV gerado.")


Carregados 10 resultados existentes (resume).
  PROCESSOS: 349 total | 339 novos | 10 ja feitos

  BATCH 1/34  (10 processos)


batch 1:   0%|          | 0/10 [00:00<?, ?it/s]

  [ok] process_0056238-72.2011.8.26.0222      0 | 0 | 1     15.8s    12.9k tok  1x
  [ok] process_0051499-22.2012.8.26.0222      0 | 0 | 1     18.4s    14.6k tok  1x
  [ok] process_0053886-81.2012.8.26.0651      0 | 0 | 1     17.6s    13.1k tok  1x
  [ok] process_0004795-62.2007.8.26.0274      0 | 0 | 3      9.2s     6.8k tok  1x
  [ok] process_1000014-76.2017.8.26.0355      1 | 0 | 1     18.5s    12.7k tok  1x
  [ok] process_0004312-59.2013.8.26.0294      0 | 0 | 3      7.0s     4.2k tok  1x
  [ok] process_1000015-85.2019.8.26.0292      0 | 0 | 3      7.8s     5.6k tok  1x
  [ok] process_1000019-98.2017.8.26.0355      1 | 0 | 1     17.6s    12.4k tok  1x
  [ok] process_1000047-84.2019.8.26.0294      0 | 0 | 1     21.0s    13.7k tok  1x
  [ok] process_1000043-58.2019.8.26.0355      0 | 0 | 3     13.9s     8.6k tok  1x

  Batch salvo -> /content/drive/MyDrive/process_classification_results.csv
  Tempo do batch 1: 146.9s
----------------------------------------------------------------
  

batch 2:   0%|          | 0/10 [00:00<?, ?it/s]

  [ok] process_1000036-51.2019.8.26.0651      0 | 0 | 3     10.6s     8.0k tok  1x
  [ok] process_1000117-15.2019.8.26.0355      1 | 0 | 1     20.5s    11.8k tok  1x
  [ok] process_1000068-70.2020.8.26.0441      0 | 0 | 1     15.6s     9.8k tok  1x
